In [365]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

In [366]:
df = pd.read_csv(
    "/Users/jw/Desktop/MIDS SP25/Machine Learning/Final Project/Restaurant_and_Market_Health_Violations_20250428.csv",
    encoding="latin-1",
)
df = df[["facility_zip", "facility_address", "owner_name", "grade"]]
df = df.rename(
    columns={
        "facility_zip": "FACULTY ZIP",
        "facility_address": "FACULTY ADDRESS",
        "owner_name": "OWNER NAME",
        "grade": "GRADE",
    }
)
df = df.dropna()

In [367]:
df.head()

,FACULTY ZIP,FACULTY ADDRESS,OWNER NAME,GRADE
0,90027,5151 HOLLYWOOD BLVD,5151 HOLLYWOOD LLC,A
1,90027,5151 HOLLYWOOD BLVD,5151 HOLLYWOOD LLC,A
2,90027,5151 HOLLYWOOD BLVD,5151 HOLLYWOOD LLC,A
3,90027,5151 HOLLYWOOD BLVD,5151 HOLLYWOOD LLC,A
4,90027,5151 HOLLYWOOD BLVD,5151 HOLLYWOOD LLC,A


In [368]:
X = df[["OWNER NAME"]]
y = df[["GRADE"]]

In [369]:
# Splitting train and test datasets

from sklearn.model_selection import train_test_split

train_data, test_data, train_labels, test_labels = train_test_split(
    X, y, random_state=100
)

print("Shape of train {}, shape of test {}".format(train_data.shape, test_data.shape))

Shape of train (235213, 1), shape of test (78405, 1)


In [370]:
import pandas as pd
import numpy as np
from collections import defaultdict
import re

In [371]:
def preprocess_string(str_arg):
    """ "
    Parameters:
    ----------
    str_arg: example string to be preprocessed

    What the function does?
    -----------------------
    Preprocess the string argument - str_arg - such that :
    1. everything apart from letters is excluded
    2. multiple spaces are replaced by single space
    3. str_arg is converted to lower case

    Example:
    --------
    Input :  Menu is absolutely perfect,loved it!
    Output:  ['menu', 'is', 'absolutely', 'perfect', 'loved', 'it']

    Returns:
    ---------
    Preprocessed string

    """

    cleaned_str = re.sub(
        "[^a-z\s]+", " ", str_arg, flags=re.IGNORECASE
    )  # every char except alphabets is replaced
    cleaned_str = re.sub(
        "(\s+)", " ", cleaned_str
    )  # multiple spaces are replaced by single space
    cleaned_str = cleaned_str.lower()  # converting the cleaned string to lower case

    return cleaned_str  # returning the preprocessed string

<>:26: SyntaxWarning: invalid escape sequence '\s'
<>:29: SyntaxWarning: invalid escape sequence '\s'
<>:26: SyntaxWarning: invalid escape sequence '\s'
<>:29: SyntaxWarning: invalid escape sequence '\s'
/var/folders/xm/yhmrwng57qq9wlcywlm89pxh0000gn/T/ipykernel_46775/3024209837.py:26: SyntaxWarning: invalid escape sequence '\s'
  "[^a-z\s]+", " ", str_arg, flags=re.IGNORECASE
/var/folders/xm/yhmrwng57qq9wlcywlm89pxh0000gn/T/ipykernel_46775/3024209837.py:29: SyntaxWarning: invalid escape sequence '\s'
  "(\s+)", " ", cleaned_str


## 1. Simple Naive Bayes

In [372]:
class NaiveBayes:

    def __init__(self, unique_classes):

        self.classes = unique_classes  # Constructor is sinply passed with unique number of classes of the training set

    def addToBow(self, example, dict_index):
        """
        Parameters:
        1. example
        2. dict_index - implies to which BoW category this example belongs to
        What the function does?
        -----------------------
        It simply splits the example on the basis of space as a tokenizer and adds every tokenized word to
        its corresponding dictionary/BoW
        Returns:
        ---------
        Nothing

        """

        if isinstance(example, np.ndarray):
            example = example[0]

        for token_word in example.split():  # for every word in preprocessed example

            self.bow_dicts[dict_index][token_word] += 1  # increment in its count

    def train(self, dataset, labels):
        """
        Parameters:
        1. dataset - shape = (m X d)
        2. labels - shape = (m,)
        What the function does?
        -----------------------
        This is the training function which will train the Naive Bayes Model i.e compute a BoW for each
        category/class.
        Returns:
        ---------
        Nothing

        """

        self.examples = dataset
        self.labels = labels
        self.bow_dicts = np.array(
            [defaultdict(lambda: 0) for index in range(self.classes.shape[0])]
        )

        # only convert to numpy arrays if initially not passed as numpy arrays - else its a useless recomputation

        if not isinstance(self.examples, np.ndarray):
            self.examples = np.array(self.examples)
        if not isinstance(self.labels, np.ndarray):
            self.labels = np.array(self.labels)

        # constructing BoW for each category
        for cat_index, cat in enumerate(self.classes):

            all_cat_examples = self.examples[
                self.labels == cat
            ]  # filter all examples of category == cat

            # get examples preprocessed

            cleaned_examples = [
                preprocess_string(cat_example) for cat_example in all_cat_examples
            ]

            cleaned_examples = pd.DataFrame(data=cleaned_examples)

            # now costruct BoW of this particular category
            np.apply_along_axis(self.addToBow, 1, cleaned_examples, cat_index)

        ###################################################################################################

        """
            Although we are done with the training of Naive Bayes Model BUT!!!!!!
            ------------------------------------------------------------------------------------
            Remember The Test Time Forumla ? : {for each word w [ count(w|c)+1 ] / [ count(c) + |V| + 1 ] } * p(c)
            ------------------------------------------------------------------------------------
            
            We are done with constructing of BoW for each category. But we need to precompute a few 
            other calculations at training time too:
            1. prior probability of each class - p(c)
            2. vocabulary |V| 
            3. denominator value of each class - [ count(c) + |V| + 1 ] 
            
            Reason for doing this precomputing calculations stuff ???
            ---------------------
            We can do all these 3 calculations at test time too BUT doing so means to re-compute these 
            again and again every time the test function will be called - this would significantly
            increase the computation time especially when we have a lot of test examples to classify!!!).  
            And moreover, it doensot make sense to repeatedly compute the same thing - 
            why do extra computations ???
            So we will precompute all of them & use them during test time to speed up predictions.
            
        """

        ###################################################################################################

        prob_classes = np.empty(self.classes.shape[0])
        all_words = []
        cat_word_counts = np.empty(self.classes.shape[0])
        for cat_index, cat in enumerate(self.classes):

            # Calculating prior probability p(c) for each class
            prob_classes[cat_index] = np.sum(self.labels == cat) / float(
                self.labels.shape[0]
            )

            # Calculating total counts of all the words of each class
            count = list(self.bow_dicts[cat_index].values())
            cat_word_counts[cat_index] = (
                np.sum(np.array(list(self.bow_dicts[cat_index].values()))) + 1
            )  # |v| is remaining to be added

            # get all words of this category
            all_words += self.bow_dicts[cat_index].keys()

        # combine all words of every category & make them unique to get vocabulary -V- of entire training set

        self.vocab = np.unique(np.array(all_words))
        self.vocab_length = self.vocab.shape[0]

        # computing denominator value
        denoms = np.array(
            [
                cat_word_counts[cat_index] + self.vocab_length + 1
                for cat_index, cat in enumerate(self.classes)
            ]
        )

        """
            Now that we have everything precomputed as well, its better to organize everything in a tuple 
            rather than to have a separate list for every thing.
            
            Every element of self.cats_info has a tuple of values
            Each tuple has a dict at index 0, prior probability at index 1, denominator value at index 2
        """

        self.cats_info = [
            (self.bow_dicts[cat_index], prob_classes[cat_index], denoms[cat_index])
            for cat_index, cat in enumerate(self.classes)
        ]
        self.cats_info = np.array(self.cats_info)

    def getExampleProb(self, test_example):
        """
        Parameters:
        -----------
        1. a single test example
        What the function does?
        -----------------------
        Function that estimates posterior probability of the given test example
        Returns:
        ---------
        probability of test example in ALL CLASSES
        """

        likelihood_prob = np.zeros(
            self.classes.shape[0]
        )  # to store probability w.r.t each class

        # finding probability w.r.t each class of the given test example
        for cat_index, cat in enumerate(self.classes):

            for (
                test_token
            ) in (
                test_example.split()
            ):  # split the test example and get p of each test word

                ####################################################################################

                # This loop computes : for each word w [ count(w|c)+1 ] / [ count(c) + |V| + 1 ]

                ####################################################################################

                # get total count of this test token from it's respective training dict to get numerator value
                test_token_counts = self.cats_info[cat_index][0].get(test_token, 0) + 1

                # now get likelihood of this test_token word
                test_token_prob = test_token_counts / float(
                    self.cats_info[cat_index][2]
                )

                # remember why taking log? To prevent underflow!
                likelihood_prob[cat_index] += np.log(test_token_prob)

        # we have likelihood estimate of the given example against every class but we need posterior probility
        post_prob = np.empty(self.classes.shape[0])
        for cat_index, cat in enumerate(self.classes):
            post_prob[cat_index] = likelihood_prob[cat_index] + np.log(
                self.cats_info[cat_index][1]
            )

        return post_prob

    def test(self, test_set):
        """
        Parameters:
        -----------
        1. A complete test set of shape (m,)

        What the function does?
        -----------------------
        Determines probability of each test example against all classes and predicts the label
        against which the class probability is maximum
        Returns:
        ---------
        Predictions of test examples - A single prediction against every test example
        """

        predictions = []  # to store prediction of each test example
        for example in test_set:

            # preprocess the test example the same way we did for training set exampels
            cleaned_example = preprocess_string(example)

            # simply get the posterior probability of every example
            post_prob = self.getExampleProb(
                cleaned_example
            )  # get prob of this example for both classes

            # simply pick the max value and map against self.classes!
            predictions.append(self.classes[np.argmax(post_prob)])

        return np.array(predictions)

In [373]:
print(set(type(label) for label in train_labels))

{<class 'str'>}


In [374]:
nb = NaiveBayes(np.unique(train_labels))  # instantiate a NB class object
print("---------------- Training In Progress --------------------")

nb.train(train_data, train_labels)  # start tarining by calling the train function
print("----------------- Training Completed ---------------------")

---------------- Training In Progress --------------------
----------------- Training Completed ---------------------


In [375]:
pclasses = nb.test(test_data)  # get predcitions for test set

# check how many predcitions actually match original test labels
test_acc = np.sum(pclasses == test_labels) / float(test_labels.shape[0])

print(
    "Test Set Examples: ", test_labels.shape[0]
)  # Outputs : Test Set Examples:   78405
print(
    "Test Set Accuracy: ", test_acc * 100, "%"
)  # Outputs : Test Set Accuracy:   86.377144 %

Test Set Examples:  78405
Test Set Accuracy:  GRADE    86.377144
dtype: float64 %


/opt/anaconda3/lib/python3.12/site-packages/numpy/core/fromnumeric.py:86: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)


## 2. Multinomial Naive Bayes (Only Owner Name Variable)

In [376]:
df["GRADE"] = df["GRADE"].map({"A": 1, "B": 2, "C": 3, "D": 4})

In [377]:
X = df["OWNER NAME"]
Y = df["GRADE"]

In [378]:
cv = CountVectorizer()
X = cv.fit_transform(X)

In [379]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.33, random_state=42
)

In [380]:
# Naive Bayes Classifier
clf = MultinomialNB()
clf.fit(X_train, y_train)
clf.score(X_test, y_test)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.91      0.94      0.93     89302
           2       0.52      0.41      0.46     12741
           3       0.36      0.26      0.30      1451

    accuracy                           0.87    103494
   macro avg       0.60      0.54      0.56    103494
weighted avg       0.86      0.87      0.86    103494



# 3. Running various models on additional variables + owners name 

In [ ]:
df1 = pd.read_csv(
    "/Users/jw/Desktop/MIDS SP25/Machine Learning/Final Project/Restaurant_and_Market_Health_Violations_20250428.csv",
    encoding="latin-1",
)
df1 = df1[["facility_zip", "facility_address", "facility_name", "owner_name", "grade"]]
df1 = df1.rename(
    columns={
        "facility_zip": "FACILITY ZIP",
        "facility_address": "FACILITY ADDRESS",
        "owner_name": "OWNER NAME",
        "grade": "GRADE",
        "facility_name": "FACILITY NAME",
    }
)

In [ ]:
df1

,FACILITY ZIP,FACILITY ADDRESS,FACILITY NAME,OWNER NAME,GRADE
0,90027,5151 HOLLYWOOD BLVD,KRUANG TEDD,5151 HOLLYWOOD LLC,A
1,90027,5151 HOLLYWOOD BLVD,KRUANG TEDD,5151 HOLLYWOOD LLC,A
2,90027,5151 HOLLYWOOD BLVD,KRUANG TEDD,5151 HOLLYWOOD LLC,A
3,90027,5151 HOLLYWOOD BLVD,KRUANG TEDD,5151 HOLLYWOOD LLC,A
4,90027,5151 HOLLYWOOD BLVD,KRUANG TEDD,5151 HOLLYWOOD LLC,A
...,...,...,...,...,...
313670,90025,2012 SAWTELLE BLVD,TEN TEN YU RAMEN,"WAO, INC.",A
313671,90025,2012 SAWTELLE BLVD,TEN TEN YU RAMEN,"WAO, INC.",A
313672,90063,3600 E CESAR E CHAVEZ AVE,SUPERIOR GROCERS #113,"SUPER CENTER CONCEPTS, INC",A
313673,90063,3600 E CESAR E CHAVEZ AVE,SUPERIOR GROCERS #113,"SUPER CENTER CONCEPTS, INC",A


In [ ]:
df1["GRADE"].unique()

array(['A', 'B', 'C', nan], dtype=object)

In [ ]:
df1 = df1.dropna()
df1.isnull().sum()

FACILITY ZIP        0
FACILITY ADDRESS    0
FACILITY NAME       0
OWNER NAME          0
GRADE               0
dtype: int64

In [ ]:
df1["GRADE"] = df1["GRADE"].map({"A": 1, "B": 2, "C": 3})

/var/folders/xm/yhmrwng57qq9wlcywlm89pxh0000gn/T/ipykernel_46775/1602951335.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1["GRADE"] = df1["GRADE"].map({"A": 1, "B": 2, "C": 3})


In [ ]:
X = df1["OWNER NAME"]
X1 = df1["FACILITY ADDRESS"]
X2 = df1["FACILITY ZIP"]
X3 = df1["FACILITY NAME"]
Y = df1["GRADE"]

In [ ]:
df1.head()

,FACILITY ZIP,FACILITY ADDRESS,FACILITY NAME,OWNER NAME,GRADE
0,90027,5151 HOLLYWOOD BLVD,KRUANG TEDD,5151 HOLLYWOOD LLC,1
1,90027,5151 HOLLYWOOD BLVD,KRUANG TEDD,5151 HOLLYWOOD LLC,1
2,90027,5151 HOLLYWOOD BLVD,KRUANG TEDD,5151 HOLLYWOOD LLC,1
3,90027,5151 HOLLYWOOD BLVD,KRUANG TEDD,5151 HOLLYWOOD LLC,1
4,90027,5151 HOLLYWOOD BLVD,KRUANG TEDD,5151 HOLLYWOOD LLC,1


In [ ]:
df1["GRADE"].unique()

array([1, 2, 3])

In [ ]:
grade1_index = Y[Y.values == 1].index
grade2_index = Y[Y.values == 2].index
grade3_index = Y[Y.values == 3].index

highest = grade1_index
higher = grade2_index
lower = grade3_index

In [ ]:
higher

Index([   227,    228,    229,    230,    231,    232,    233,    234,    235,
          236,
       ...
       313482, 313483, 313484, 313485, 313495, 313496, 313497, 313498, 313499,
       313500],
      dtype='int64', length=38345)

In [ ]:
# remember higher is a list of indexes, either of 0 or 1's in the response variable in training set
higher = np.random.choice(higher, size=5 * len(lower))
highest = np.random.choice(highest, size=8 * len(lower))

lower = np.asarray(lower)

new_indexes = np.concatenate((lower, higher, highest))

X = X.loc[new_indexes,]
X1 = X1.loc[new_indexes,]
X2 = X2.loc[new_indexes,]
X3 = X3.loc[new_indexes,]
Y = Y.loc[new_indexes]

In [ ]:
DataFrame123 = pd.concat([X, X1, X2, X3, Y], axis=1)

In [ ]:
DataFrame123.head()

,OWNER NAME,FACILITY ADDRESS,FACILITY ZIP,FACILITY NAME,GRADE
331,L & J CUISINE INC.,4578 WHITTIER BLVD,90022-2430,CHUNG KING CAFE HOUSE,3
332,L & J CUISINE INC.,4578 WHITTIER BLVD,90022-2430,CHUNG KING CAFE HOUSE,3
333,L & J CUISINE INC.,4578 WHITTIER BLVD,90022-2430,CHUNG KING CAFE HOUSE,3
334,L & J CUISINE INC.,4578 WHITTIER BLVD,90022-2430,CHUNG KING CAFE HOUSE,3
335,L & J CUISINE INC.,4578 WHITTIER BLVD,90022-2430,CHUNG KING CAFE HOUSE,3


In [ ]:
DataFrame123["GRADE"].unique()

array([3, 2, 1])

In [ ]:
DataFrame123["FACILITY ZIP"] = DataFrame123["FACILITY ZIP"].astype(str)

In [ ]:
# X=df1[['FACILITY ZIP','FACILITY ADDRESS']]
X = DataFrame123["FACILITY ADDRESS"]
Y = DataFrame123["GRADE"]

In [ ]:
X.tail()

140092    404 S FIGUEROA ST LBBY LOBBY
28804                  574 HILGARD AVE
232491       12003 AVALON BLVD STE 106
215393          2273 W WASHINGTON BLVD
71428                 2596 W PICO BLVD
Name: FACILITY ADDRESS, dtype: object

In [ ]:
cv = CountVectorizer()
X = cv.fit_transform(X)
X

<59612x5479 sparse matrix of type '<class 'numpy.int64'>'
	with 202270 stored elements in Compressed Sparse Row format>

In [ ]:
# X=df1[['FACILITY ZIP','FACILITY ADDRESS']]
X1 = DataFrame123["FACILITY ZIP"]

In [ ]:
cv = CountVectorizer()
X1 = cv.fit_transform(X1)
X1

<59612x748 sparse matrix of type '<class 'numpy.int64'>'
	with 63497 stored elements in Compressed Sparse Row format>

In [ ]:
# X=df1[['FACILITY ZIP','FACILITY ADDRESS']]
X2 = DataFrame123["OWNER NAME"]

In [ ]:
cv = CountVectorizer()
X2 = cv.fit_transform(X2)
X2

<59612x9163 sparse matrix of type '<class 'numpy.int64'>'
	with 173471 stored elements in Compressed Sparse Row format>

In [ ]:
# X=df1[['FACILITY ZIP','FACILITY ADDRESS']]
X3 = DataFrame123["FACILITY NAME"]

In [ ]:
cv = CountVectorizer()
X3 = cv.fit_transform(X3)
X3

<59612x7550 sparse matrix of type '<class 'numpy.int64'>'
	with 158770 stored elements in Compressed Sparse Row format>

In [ ]:
from scipy.sparse import hstack

Comb = hstack((X2, X3))
comb = hstack((Comb, X))

In [ ]:
Combined = hstack((comb, X1))

In [ ]:
Combined

<59612x22940 sparse matrix of type '<class 'numpy.int64'>'
	with 598008 stored elements in Compressed Sparse Row format>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X3, Y, test_size=0.27, random_state=40
)

In [ ]:
# Naive Bayes Classifier
clf = MultinomialNB(alpha=1.0, class_prior=None, fit_prior=True)
clf.fit(X_train, y_train)
clf.score(X_test, y_test)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))
a = classification_report(y_test, y_pred)

              precision    recall  f1-score   support

           1       0.79      0.78      0.79      9179
           2       0.64      0.68      0.66      5762
           3       0.60      0.44      0.51      1155

    accuracy                           0.72     16096
   macro avg       0.68      0.63      0.65     16096
weighted avg       0.72      0.72      0.72     16096



In [ ]:
X_train_prob = pd.DataFrame(clf.predict_proba(X_train))
X_test_prob = pd.DataFrame(clf.predict_proba(X_test))
X1_train_prob = pd.DataFrame(clf.predict_proba(X_train))
X1_test_prob = pd.DataFrame(clf.predict_proba(X_test))
X2_train_prob = pd.DataFrame(clf.predict_proba(X_train))
X2_test_prob = pd.DataFrame(clf.predict_proba(X_test))
X3_train_prob = pd.DataFrame(clf.predict_proba(X_train))
X3_test_prob = pd.DataFrame(clf.predict_proba(X_test))

In [ ]:
X_test_prob.shape

(16096, 3)

In [ ]:
y_test.shape

(16096,)

In [ ]:
X1_test_prob.shape

(16096, 3)

In [ ]:
X2_train_prob.shape

(43516, 3)

In [ ]:
X3_train_prob.shape

(43516, 3)

In [ ]:
# a_test = pd.concat([X_train_prob, X1_train_prob, X2_train_prob, X3_train_prob], axis=1)
# b_test = pd.concat([X_test_prob, X1_test_prob, X2_test_prob, X3_test_prob], axis=1)

In [ ]:
# a_train = pd.concat([a_test, y_train])

In [ ]:
# a_test = pd.concat(
#     [X_train_prob, X1_train_prob, X2_train_prob, X3_train_prob, y_train], axis=1
# )
# b_test = pd.concat(
#     [X_test_prob, X1_test_prob, X2_test_prob, X3_test_prob, y_test], axis=1
# )

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

lreg_clf = LogisticRegression()

param_grid = {"penalty": ["l1", "l2"]}

grid_search = GridSearchCV(lreg_clf, param_grid, cv=5, return_train_score=True)
grid_search.fit(X_train_prob, y_train)

/opt/anaconda3/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:547: FitFailedWarning: 
5 fits failed out of a total of 10.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 895, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/opt/anaconda3/lib/python3.12/site-packages/sklearn/base.py", line 1474, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py", line 1172, in fit
    solver =

GridSearchCV(cv=5, estimator=LogisticRegression(),
             param_grid={'penalty': ['l1', 'l2']}, return_train_score=True)

In [ ]:
grid_search.best_score_

0.7592609525282018

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, f1_score

lreg_clf = LogisticRegression(penalty="l2")
lreg_clf.fit(X_train_prob, y_train)
results = pd.DataFrame(index=None, columns=["model"])
y_lreg_clf = lreg_clf.predict(X_test_prob)
train_Rsquare = lreg_clf.score(X_train_prob, y_train)
test_Rsquare = lreg_clf.score(X_test_prob, y_test)
train_MSE = mean_squared_error(y_train, lreg_clf.predict(X_train_prob))
test_MSE = mean_squared_error(y_test, lreg_clf.predict(X_test_prob))
f1_score_train = f1_score(y_train, lreg_clf.predict(X_train_prob), average="weighted")
f1_score_test = f1_score(y_test, lreg_clf.predict(X_test_prob), average="weighted")
results = pd.concat(
    [
        results,
        pd.DataFrame(
            [
                {
                    "model": "Logistic Regression",  # fix name also!
                    "train_Rsquare": train_Rsquare,
                    "test_Rsquare": test_Rsquare,
                    "train_MSE": train_MSE,
                    "test_MSE": test_MSE,
                    "f1_score_train": f1_score_train,
                    "f1_score_test": f1_score_test,
                }
            ]
        ),
    ],
    ignore_index=True,
)
results

,model,train_Rsquare,test_Rsquare,train_MSE,test_MSE,f1_score_train,f1_score_test
0,Logistic Regression,0.759215,0.721173,0.307519,0.358598,0.755623,0.718045


In [ ]:
lreg_clf.score(X_train_prob, y_train)

0.7592150013788032

In [ ]:
lreg_clf.score(X_test_prob, y_test)

0.7211729622266402

In [ ]:
mean_squared_error(y_train, lreg_clf.predict(X_train_prob))

0.3075190734442504

In [ ]:
mean_squared_error(y_test, lreg_clf.predict(X_test_prob))

0.3585984095427435

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_clf = DecisionTreeClassifier()
param_grid = {"max_depth": [5, 10, 20, 50, 100]}

grid_search = GridSearchCV(dt_clf, param_grid, cv=5, return_train_score=True)
grid_search.fit(X_train_prob, y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'max_depth': [5, 10, 20, 50, 100]},
             return_train_score=True)

In [ ]:
grid_search.best_params_

{'max_depth': 50}

In [ ]:
grid_search.best_score_

0.8009468031223175

In [ ]:
results = pd.DataFrame(index=None, columns=["model"])
dt_clf = DecisionTreeClassifier(max_depth=10)
dt_clf.fit(X_train_prob, y_train)
y_dt_clf = dt_clf.predict(X_test_prob)
train_Rsquare = dt_clf.score(X_train_prob, y_train)
test_Rsquare = dt_clf.score(X_test_prob, y_test)
train_MSE = mean_squared_error(y_train, dt_clf.predict(X_train_prob))
test_MSE = mean_squared_error(y_test, dt_clf.predict(X_test_prob))
results = pd.concat(
    [
        results,
        pd.DataFrame(
            [
                {
                    "model": "Decision Tree Classifier",
                    "train_Rsquare": train_Rsquare,
                    "test_Rsquare": test_Rsquare,
                    "train_MSE": train_MSE,
                    "test_MSE": test_MSE,
                }
            ]
        ),
    ],
    ignore_index=True,
)

results

,model,train_Rsquare,test_Rsquare,train_MSE,test_MSE
0,Decision Tree Classifier,0.791042,0.751615,0.257354,0.306909


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint as sp_randint

# Tuning ridge on new dataset
param_grid = {
    "max_depth": [3, 5],
    "min_samples_split": sp_randint(2, 30),
    "min_samples_leaf": sp_randint(1, 20),
    "bootstrap": [True, False],
}
random_search = RandomizedSearchCV(
    RandomForestClassifier(n_estimators=1000),
    param_distributions=param_grid,
    n_iter=20,
    random_state=0,
    n_jobs=-1,
    return_train_score=True,
)
random_search.fit(X_train_prob, y_train)

RandomizedSearchCV(estimator=RandomForestClassifier(n_estimators=1000),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'bootstrap': [True, False],
                                        'max_depth': [3, 5],
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x33f429820>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x33f61bc80>},
                   random_state=0, return_train_score=True)

In [ ]:
random_search.best_params_

{'bootstrap': True,
 'max_depth': 5,
 'min_samples_leaf': 17,
 'min_samples_split': 21}

In [ ]:
random_search.best_score_

0.7664307407942494

In [ ]:
rf_clf = RandomForestClassifier(
    bootstrap=True, max_depth=5, min_samples_leaf=1, min_samples_split=5
)
rf_clf.fit(X_train_prob, y_train)
y_rf_clf = rf_clf.predict(X_test_prob)
train_Rsquare = rf_clf.score(X_train_prob, y_train)
test_Rsquare = rf_clf.score(X_test_prob, y_test)
train_MSE = mean_squared_error(y_train, rf_clf.predict(X_train_prob))
test_MSE = mean_squared_error(y_test, rf_clf.predict(X_test_prob))
results = pd.concat(
    [
        results,
        pd.DataFrame(
            [
                {
                    "model": "Random Forest Classifier",
                    "train_Rsquare": train_Rsquare,
                    "test_Rsquare": test_Rsquare,
                    "train_MSE": train_MSE,
                    "test_MSE": test_MSE,
                }
            ]
        ),
    ],
    ignore_index=True,
)
results

,model,train_Rsquare,test_Rsquare,train_MSE,test_MSE
0,Decision Tree Classifier,0.791042,0.751615,0.257354,0.306909
1,Random Forest Classifier,0.768085,0.728504,0.291548,0.347540


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.5, random_state=0)
gb.fit(X_train_prob, y_train)
print(gb.score(X_train_prob, y_train))
print(gb.score(X_test_prob, y_test))

0.8070364923246622
0.7621769383697813
